# Developmental LM — formal 1000-item Meaning probe
This notebook never connects to Google Drive. It uses the repository's fixed 1000-item Meaning TSV; upload only the Phase 1 checkpoint ZIP. It trains on a T4 GPU, keeps all artifacts under `/content`, creates `meaning_probe_results.zip`, and downloads it once at the end.

In [ ]:
import pathlib, shutil, subprocess, sys
REPO_URL = 'https://github.com/ss-sebastian/developmental_checkpoints_word_recognition.git'
PROJECT = pathlib.Path('/content/developmental_checkpoints_word_recognition')
if PROJECT.exists(): shutil.rmtree(PROJECT)
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT)], check=True)
SOURCE_DIR = PROJECT / 'src'
if str(SOURCE_DIR) not in sys.path: sys.path.insert(0, str(SOURCE_DIR))
print('Installed commit:', subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
import torch
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > T4 GPU, then rerun.'
print('CUDA:', torch.cuda.get_device_name(0), '| PyTorch:', torch.__version__)

In [ ]:
from google.colab import files
UPLOAD_DIR = pathlib.Path('/content/devlm_uploads'); UPLOAD_DIR.mkdir(exist_ok=True)
STIMULUS_PATH = PROJECT / 'data' / 'task_adaptation_stimuli' / 'meaning_1000' / 'final_all.tsv'
assert STIMULUS_PATH.is_file(), f'Missing bundled formal dataset: {STIMULUS_PATH}'
print('Using fixed formal stimuli:', STIMULUS_PATH)

In [ ]:
import zipfile
print('Upload the ZIP containing exactly 30 Phase 1 checkpoints')
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one checkpoint ZIP and rerun this cell.'
name, content = next(iter(uploaded.items()))
checkpoint_zip = UPLOAD_DIR / pathlib.Path(name).name; checkpoint_zip.write_bytes(content)
CHECKPOINT_DIR = pathlib.Path('/content/devlm_checkpoints')
if CHECKPOINT_DIR.exists(): shutil.rmtree(CHECKPOINT_DIR)
CHECKPOINT_DIR.mkdir()
with zipfile.ZipFile(checkpoint_zip) as archive:
    root = CHECKPOINT_DIR.resolve()
    for member in archive.infolist():
        destination = (CHECKPOINT_DIR / member.filename).resolve()
        if root not in destination.parents and destination != root: raise ValueError(f'Unsafe ZIP member: {member.filename}')
    archive.extractall(CHECKPOINT_DIR)
print('Extracted:', CHECKPOINT_DIR)

In [ ]:
import importlib, devlm
importlib.invalidate_caches()
from devlm.adaptation.meaning_probe import load_meaning_manifest
from devlm.adaptation.train import discover_checkpoints, discover_feature_table
items = load_meaning_manifest(STIMULUS_PATH)
checkpoints = discover_checkpoints(CHECKPOINT_DIR)
try: FEATURE_TABLE_PATH = discover_feature_table(CHECKPOINT_DIR)
except ValueError: FEATURE_TABLE_PATH = PROJECT / 'colab' / 'ipa_feature_mapping.json'
print('Code:', pathlib.Path(devlm.__file__).resolve())
print('Validated items:', len(items), '| checkpoints:', len(checkpoints))
print('Feature table:', FEATURE_TABLE_PATH)

In [ ]:
OUTPUT_DIR = pathlib.Path('/content/meaning_probe_run')
cmd = [sys.executable, '-u', '-m', 'devlm.adaptation.meaning_probe_cli',
       '--stimuli', str(STIMULUS_PATH), '--checkpoints-dir', str(CHECKPOINT_DIR),
       '--feature-table', str(FEATURE_TABLE_PATH), '--output-dir', str(OUTPUT_DIR),
       '--device', 'cuda', '--learning-rate', '0.001', '--weight-decay', '0',
       '--batch-size', '32', '--encoding-batch-size', '128', '--max-epochs', '100',
       '--patience', '10', '--min-delta', '0.0001', '--noise-sigma', '0.05',
       '--input-noise-seed', '20260904', '--initialization-seed', '1729']
print('Starting formal Meaning probe...', flush=True)
subprocess.run(cmd, cwd=PROJECT, check=True)

In [ ]:
import csv, json
with (OUTPUT_DIR / 'summary' / 'run_status.csv').open() as handle: status = list(csv.DictReader(handle))
print('Success:', sum(x['status']=='success' for x in status), '/ 30')
failed = [x for x in status if x['status'] != 'success']
if failed: print('Failures:', failed)
assert len(status) == 30 and not failed, 'Run is incomplete; inspect run_log.txt before downloading.'
bundle = pathlib.Path('/content/meaning_probe_results.zip')
assert bundle.is_file(), bundle
print('Bundle:', bundle, f'{bundle.stat().st_size/1e6:.1f} MB')
files.download(str(bundle))